## 高阶函数

高阶函数英文叫Higher-order function。什么是高阶函数？我们以实际代码为例子，一步一步深入概念。

### 变量可以指向函数
以Python内置的求绝对值的函数abs()为例，调用该函数用以下代码：

In [2]:
f = abs
f

<function abs(x, /)>

In [3]:
print(f(-2))

2


函数名也是变量

那么函数名是什么呢？函数名其实就是指向函数的变量！对于abs()这个函数，完全可以把函数名abs看成变量，它指向一个可以计算绝对值的函数！

如果把abs指向其他对象，会有什么情况发生？

In [4]:
abs = 123

abs(-123)

TypeError: 'int' object is not callable

把abs指向10后，就无法通过abs(-10)调用该函数了！因为abs这个变量已经不指向求绝对值函数而是指向一个整数10！

当然实际代码绝对不能这么写，这里是为了说明函数名也是变量。要恢复abs函数，请重启Python交互环境。

注：由于abs函数实际上是定义在import builtins模块中的，所以要让修改abs变量的指向在其它模块也生效，要用import builtins; builtins.abs = 10。

In [ ]:
import builtins

builtins.abs(-123)
abs = builtins.abs

<function abs(x, /)>

### 传入函数

既然变量可以指向函数，函数的参数能接收变量，那么一个函数就可以接收另一个函数作为参数，这种函数就称之为高阶函数。

一个最简单的高阶函数：

In [8]:
def add(x, y, f):
    return f(x) + f(y)

当我们调用add(-5, 6, abs)时，参数x，y和f分别接收-5，6和abs，根据函数定义，我们可以推导计算过程为：

In [ ]:
print(add(-5, 6, abs))

11


编写高阶函数，就是让函数的参数能够接收别的函数。

### map/reduce

Python内建了map()和reduce()函数。

如果你读过Google的那篇大名鼎鼎的论文“MapReduce: Simplified Data Processing on Large Clusters”，你就能大概明白map/reduce的概念。

In [19]:
def f(x):
    return x*x;

r = map(f, [2, 3, 4, 6])
print(r)
print(list(r))

[4, 9, 16, 36]


map()作为高阶函数，事实上它把运算规则抽象了，因此，我们不但可以计算简单的f(x)=x2，还可以计算任意复杂的函数，比如，把这个list所有数字转为字符串：

In [ ]:
list(map(str, [1, 2, 3, 4, 5, 6, 7, 8, 9])) # str()函数

['1', '2', '3', '4', '5', '6', '7', '8', '9']

reduce的用法。reduce把一个函数作用在一个序列[x1, x2, x3, ...]上，这个函数必须接收两个参数，reduce把结果继续和序列的下一个元素做累积计算，其效果就是：

reduce(f, [x1, x2, x3, x4]) = f(f(f(x1, x2), x3), x4)

In [ ]:
from functools import reduce

def add(x, y):
    return x + y # 前一个元素和后一个元素做某种计算

print(reduce(add, [1, 3, 5, 7, 9]))

25


当然求和运算可以直接用Python内建函数sum()，没必要动用reduce。

但是如果要把序列[1, 3, 5, 7, 9]变换成整数13579，reduce就可以派上用场：

In [22]:
from functools import reduce

def fn(x, y):
    return x * 10 + y

reduce(fn, [1, 3, 5, 7, 9])

13579

### filter

Python内建的filter()函数用于过滤序列。

和map()类似，filter()也接收一个函数和一个序列。和map()不同的是，filter()把传入的函数依次作用于每个元素，然后根据返回值是True还是False决定保留还是丢弃该元素。

例如，在一个list中，删掉偶数，只保留奇数，可以这么写：

In [ ]:
def is_odd(n):
    return n % 2 == 1

# 可以看出，给出的筛选条件是和结果是匹配的
list(filter(is_odd, [1, 2, 4, 5, 6, 9, 10, 15]))

[1, 5, 9, 15]

把一个序列中的空字符串删掉，可以这么写：

In [34]:
def not_empty(s):
    return s and s.strip()

print(bool('    ' and '    '.strip()))
print(bool('  1  ' and '  1  '.strip()))

list(filter(not_empty, ['A', '', 'B', None, 'C', '  ']))

False
True


['A', 'B', 'C']

可见用filter()这个高阶函数，关键在于正确实现一个“筛选”函数。

注意到filter()函数返回的是一个Iterator，也就是一个惰性序列，所以要强迫filter()完成计算结果，需要用list()函数获得所有结果并返回list。

### sorted

排序也是在程序中经常用到的算法。无论使用冒泡排序还是快速排序，排序的核心是比较两个元素的大小。如果是数字，我们可以直接比较，但如果是字符串或者两个dict呢？直接比较数学上的大小是没有意义的，因此，比较的过程必须通过函数抽象出来。

Python内置的sorted()函数就可以对list进行排序：

In [35]:
sorted([36, 5, -12, 9, -21])

[-21, -12, 5, 9, 36]

此外，sorted()函数也是一个高阶函数，它还可以接收一个key函数来实现自定义的排序，例如按绝对值大小排序：

In [36]:
sorted([36, 5, -12, 9, -21], key=abs)

[5, 9, -12, -21, 36]

key指定的函数将作用于list的每一个元素上，并根据key函数返回的结果进行排序。

现在，我们提出排序应该忽略大小写，按照字母序排序。要实现这个算法，不必对现有代码大加改动，只要我们能用一个key函数把字符串映射为忽略大小写排序即可。忽略大小写来比较两个字符串，实际上就是先把字符串都变成大写（或者都变成小写），再比较。

这样，我们给sorted传入key函数，即可实现忽略大小写的排序：

In [37]:
sorted(['bob', 'about', 'Zoo', 'Credit'], key=str.lower)

['about', 'bob', 'Credit', 'Zoo']

要进行反向排序，不必改动key函数，可以传入第三个参数reverse=True：

In [38]:
sorted(['bob', 'about', 'Zoo', 'Credit'], key=str.lower, reverse=True)

['Zoo', 'Credit', 'bob', 'about']

## 返回函数

高阶函数除了可以接受函数作为参数外，还可以把函数作为结果值返回。

我们来实现一个可变参数的求和。通常情况下，求和的函数是这样定义的：

In [1]:
def calc_sum(*args):
    ax = 0
    for n in args:
        ax = ax + n
    return ax

但是，如果不需要立刻求和，而是在后面的代码中，根据需要再计算怎么办？可以不返回求和的结果，而是返回求和的函数：

In [ ]:
def lazy_sum(*args):
    def sum():
        ax = 0
        for n in args:
            ax = ax + n
        return ax
    return sum

r_fun = lazy_sum(*list(range(1, 101))) # 从1加到100
print(r_fun)
print(r_fun())

<function lazy_sum.<locals>.sum at 0x06F42FA8>
5050


### 闭包

注意到返回的函数在其定义内部引用了局部变量args，所以，当一个函数返回了一个函数后，其内部的局部变量还被新函数引用，所以，闭包用起来简单，实现起来可不容易。

另一个需要注意的问题是，返回的函数并没有立刻执行，而是直到调用了f()才执行。我们来看一个例子：

In [6]:
def count():
    fs = []
    for i in range(1, 4):
        def f():
             return i*i
        fs.append(f)
    return fs

f1, f2, f3 = count()

在上面的例子中，每次循环，都创建了一个新的函数，然后，把创建的3个函数都返回了。

你可能认为调用f1()，f2()和f3()结果应该是1，4，9，但实际结果是：

In [7]:
print(f1(), f2(), f3())

9 9 9


全部都是9！原因就在于返回的函数引用了变量i，但它并非立刻执行。等到3个函数都返回时，它们所引用的变量i已经变成了3，因此最终结果为9。

> 返回闭包时牢记一点：返回函数不要引用任何循环变量，或者后续会发生变化的变量。

如果一定要引用循环变量怎么办？方法是再创建一个函数，用该函数的参数绑定循环变量当前的值，无论该循环变量后续如何更改，已绑定到函数参数的值不变：

In [10]:
def count():
    def f(j):
        def g():
            return j*j
        return g
    fs = []
    for i in range(1, 4):
        fs.append(f(i)) # f(i)立刻被执行，因此i的当前值被传入f()
    return fs

再看看结果：

In [12]:
f1, f2, f3 = count()
print(f1(), f2(), f3())

1 4 9


缺点是代码较长，可利用lambda函数缩短代码。

### nonlocal

使用闭包，就是内层函数引用了外层函数的局部变量。如果只是读外层变量的值，我们会发现返回的闭包函数调用一切正常：

In [13]:
def inc():
    x = 0
    def fn():
        # 仅读取x的值:
        return x + 1
    return fn

f = inc()
print(f()) # 1
print(f()) # 1


1
1


但是，如果对外层变量赋值，由于Python解释器会把x当作函数fn()的局部变量，它会报错：

In [ ]:
def inc():
    x = 0
    def fn():
        x = x + 1
        return x
    return fn

f = inc()
print(f()) # 1
print(f()) # 2

UnboundLocalError: local variable 'x' referenced before assignment

原因是x作为局部变量并没有初始化，直接计算x+1是不行的。但我们其实是想引用inc()函数内部的x，所以需要在fn()函数内部加一个nonlocal x的声明。加上这个声明后，解释器把fn()的x看作外层函数的局部变量，它已经被初始化了，可以正确计算x+1。

In [16]:
def inc():
    x = 0
    def fn():
        nonlocal x
        x = x + 1
        return x
    return fn

f = inc()
print(f()) # 1
print(f()) # 2

1
2


> 使用闭包时，对外层变量赋值前，需要先使用nonlocal声明该变量不是当前函数的局部变量。

## 匿名函数

当我们在传入函数时，有些时候，不需要显式地定义函数，直接传入匿名函数更方便。

在Python中，对匿名函数提供了有限支持。还是以map()函数为例，计算f(x)=x2时，除了定义一个f(x)的函数外，还可以直接传入匿名函数：

In [17]:
list(map(lambda x: x * x, [1, 2, 3, 4, 5, 6, 7, 8, 9]))

[1, 4, 9, 16, 25, 36, 49, 64, 81]

通过对比可以看出，匿名函数lambda x: x * x实际上就是：

In [ ]:
def f(x):
    return x * x

关键字lambda表示匿名函数，冒号前面的x表示函数参数。

匿名函数有个限制，就是只能有一个表达式，不用写return，返回值就是该表达式的结果。

用匿名函数有个好处，因为函数没有名字，不必担心函数名冲突。

In [18]:
f = lambda x: x * x
f

<function __main__.<lambda>(x)>

同样，也可以把匿名函数作为返回值返回，比如：

In [23]:
def build(x, y):
    return lambda: x * x + y * y

print(build)
build(1, 2)

<function build at 0x00E2E6A8>


<function __main__.build.<locals>.<lambda>()>

## 装饰器

由于函数也是一个对象，而且函数对象可以被赋值给变量，所以，通过变量也能调用该函数。

In [24]:
def now():
    print('2024-6-1')

f = now
f()

2024-6-1


函数对象有一个__name__属性（注意：是前后各两个下划线），可以拿到函数的名字：

In [25]:
now.__name__

'now'

现在，假设我们要增强now()函数的功能，比如，在函数调用前后自动打印日志，但又不希望修改now()函数的定义，这种在代码运行期间动态增加功能的方式，称之为“装饰器”（Decorator）。

本质上，decorator就是一个返回函数的高阶函数。所以，我们要定义一个能打印日志的decorator，可以定义如下：

In [1]:
def log(func):
    def wrapper(*args, **kw):
        print('call %s():' % func.__name__)
        return func(*args, **kw)
    return wrapper

观察上面的log，因为它是一个decorator，所以接受一个函数作为参数，并返回一个函数。我们要借助Python的@语法，把decorator置于函数的定义处：

In [5]:
@log
def now():
    print('2024-6-1')

# 调用now()函数，不仅会运行now()函数本身，还会在运行now()函数前打印一行日志：
now()

call now():
2024-6-1


把@log放到now()函数的定义处，相当于执行了语句：

In [8]:
log(now)

<function __main__.log.<locals>.wrapper(*args, **kw)>

由于log()是一个decorator，返回一个函数，所以，原来的now()函数仍然存在，只是现在同名的now变量指向了新的函数，于是调用now()将执行新函数，即在log()函数中返回的wrapper()函数。

wrapper()函数的参数定义是(*args, **kw)，因此，wrapper()函数可以接受任意参数的调用。在wrapper()函数内，首先打印日志，再紧接着调用原始函数。

如果decorator本身需要传入参数，那就需要编写一个返回decorator的高阶函数，写出来会更复杂。比如，要自定义log的文本：

In [3]:
def log(text):
    def decorator(func):
        def wrapper(*args, **kw):
            print('%s %s():' % (text, func.__name__))
            return func(*args, **kw)
        return wrapper
    return decorator

这个3层嵌套的decorator用法如下：

In [4]:
@log('execute')
def now():
    print('2024-6-1')

now()

execute now():
2024-6-1


和两层嵌套的decorator相比，3层嵌套的效果是这样的：

In [5]:
now = log('execute')(now)

我们来剖析上面的语句，首先执行log('execute')，返回的是decorator函数，再调用返回的函数，参数是now函数，返回值最终是wrapper函数。

以上两种decorator的定义都没有问题，但还差最后一步。因为我们讲了函数也是对象，它有__name__等属性，但你去看经过decorator装饰之后的函数，它们的__name__已经从原来的'now'变成了'wrapper'：

In [6]:
now.__name__

'wrapper'

因为返回的那个wrapper()函数名字就是'wrapper'，所以，需要把原始函数的__name__等属性复制到wrapper()函数中，否则，有些依赖函数签名的代码执行就会出错。

不需要编写wrapper.__name__ = func.__name__这样的代码，Python内置的functools.wraps就是干这个事的，所以，一个完整的decorator的写法如下：

In [ ]:
import functools

def log(func):
    @functools.wraps(func)
    def wrapper(*args, **kw):
        print('call %s():' % func.__name__)
        return func(*args, **kw)
    return wrapper

或者针对带参数的decorator：

In [ ]:
import functools

def log(text):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kw):
            print('%s %s():' % (text, func.__name__))
            return func(*args, **kw)
        return wrapper
    return decorator

import functools是导入functools模块。模块的概念稍候讲解。现在，只需记住在定义wrapper()的前面加上@functools.wraps(func)即可。

#### 练习

In [18]:
import time, functools

def metric2(text):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kw):
            start = time.time()
            t = func(*args, **kw)
            print('%s executed2 in %s ms, type:%s' % (func.__name__, time.time() - start, text))
            return t
        return wrapper
    return decorator

def metric(func):
    @functools.wraps(func)
    def wrapper(*args, **kw):
            start = time.time()
            t = func(*args, **kw)
            print('%s executed in %s ms' % (func.__name__, time.time() - start))
            return t
    return wrapper

# 测试
@metric
@metric2("row")
def fast(x, y):
    time.sleep(0.0012)
    return x + y;

@metric
@metric2("row")
def slow(x, y, z):
    time.sleep(0.1234)
    return x * y * z;

f = fast(11, 22)
s = slow(11, 22, 33)
if f != 33:
    print('测试失败!')
elif s != 7986:
    print('测试失败!')

fast executed2 in 0.0039980411529541016 ms, type:row
fast executed in 0.0039980411529541016 ms
slow executed2 in 0.12604880332946777 ms, type:row
slow executed in 0.12604880332946777 ms


## 偏函数

Python的functools模块提供了很多有用的功能，其中一个就是偏函数（Partial function）。要注意，这里的偏函数和数学意义上的偏函数不一样。

在介绍函数参数的时候，我们讲到，通过设定参数的默认值，可以降低函数调用的难度。而偏函数也可以做到这一点。举例如下：

int()函数可以把字符串转换为整数，当仅传入字符串时，int()函数默认按十进制转换：



In [19]:
int('12345')

12345

但int()函数还提供额外的base参数，默认值为10。如果传入base参数，就可以做N进制的转换：

In [22]:
print(int('12345', base=8))
print(int('12345', 16))

5349
74565


假设要转换大量的二进制字符串，每次都传入int(x, base=2)非常麻烦，于是，我们想到，可以定义一个int2()的函数，默认把base=2传进去：

In [23]:
def int2(x, base=2):
    return int(x, base)

这样，我们转换二进制就非常方便了：

In [24]:
int2('1000000')

64

functools.partial就是帮助我们创建一个偏函数的，不需要我们自己定义int2()，可以直接使用下面的代码创建一个新的函数int2：

In [28]:
import functools

int2 = functools.partial(int, base=2)

print(int2('1000000'))

# 自定义函数
def cprint(out, type):
    if type == 'row':
        print(out, end=" ")
    else:
        print(out)

crp = functools.partial(cprint, type = 'row')
crp(1)
crp(23)

64
1 23 

所以，简单总结functools.partial的作用就是，把一个函数的某些参数给固定住（也就是设置默认值），返回一个新的函数，调用这个新函数会更简单。

注意到上面的新的int2函数，仅仅是把base参数重新设定默认值为2，但也可以在函数调用时传入其他值：

In [29]:
int2('1000000', base=10)

1000000